# 1. Generate insertions

Construct charge-compatible singlet monomials for Yukawa couplings, the physical Higgs bilinear, and various R-parity violating operators.

Catalogue JSON inputs are missing. `USE_TEST_MODEL` selects the explicit source example; it is disabled by default. Set a small `P` for a diagnostic run. Example exports go to `results/single_higgs_example/`.

## Imports and configuration

In [ ]:
import json
import math
import time
from pathlib import Path
import numpy as np
USE_TEST_MODEL = False
P = 12
N_BATCHES = 10
EXCLUDE_CONSTANT_MU = True
pert_range = [0.1, 0.7]
non_pert_range = [0.01, 0.07]


## Charge and exponent utilities

A monomial has a nonnegative exponent vector $p$. The search retains vectors satisfying $\hat Q^T p=-\hat r$ and $\sum_a p_a\leq P$, where $\hat q=(q_1-q_0,\ldots,q_4-q_0)$. Singlet coordinates put nonperturbative fields first. Candidate exponents are streamed rather than retained in memory.

In [ ]:
def string_to_list(input_string):
    """Parse a bracketed, comma-separated integer charge vector."""
    try:
        stripped_string = input_string.strip('[]')
        if not stripped_string:
            return []
        return [int(item) for item in stripped_string.split(',')]
    except ValueError:
        raise ValueError('The input string is not in the correct format.')


In [ ]:
def generate_vectors(n, P, current_vector=None, current_sum=0):
    """Yield nonnegative exponent vectors with total degree at most P."""
    if current_vector is None:
        current_vector = []
    if len(current_vector) == n:
        yield current_vector
        return
    for exponent in range(P - current_sum + 1):
        yield from generate_vectors(n, P, current_vector + [exponent], current_sum + exponent)


In [ ]:
def linear_map(arg, flag):
    """Project five-component charges to differences from the first component.

    The input is modified in place before the redundant component is removed."""
    if flag == 'Q':
        for i in range(len(arg)):
            temp = arg[i][0]
            for j in range(f - 1):
                arg[i][j] = arg[i][j + 1] - temp
        return np.delete(arg, f - 1, 1)
    if flag == 'r':
        temp = arg[0]
        for i in range(f - 1):
            arg[i] = arg[i + 1] - temp
        return np.delete(arg, f - 1)


In [ ]:
def generate_Yukawa_charges(charges):
    """Group up/down Yukawa index pairs by their projected total charge."""
    rs_up_list = []
    rs_down_list = []
    for i in range(len(charges)):
        rs_up_temp = {}
        rs_down_temp = {}
        tens = charges[i][0]
        fives = charges[i][1]
        higgs_down = charges[i][2][0]
        higgs_up = charges[i][2][1]
        for j in range(len(tens)):
            for k in range(j, len(tens)):
                charge = linear_map(list(np.array(tens[j]) + np.array(tens[k]) + np.array(higgs_up)), 'r').tolist()
                if str(charge) in rs_up_temp:
                    rs_up_temp[str(charge)].append([j + 1, k + 1])
                else:
                    rs_up_temp[str(charge)] = [[j + 1, k + 1]]
        for j in range(len(fives)):
            for k in range(len(tens)):
                charge = linear_map(list(np.array(fives[j]) + np.array(tens[k]) + np.array(higgs_down)), 'r').tolist()
                if str(charge) in rs_down_temp:
                    rs_down_temp[str(charge)].append([k + 1, j + 1])
                else:
                    rs_down_temp[str(charge)] = [[k + 1, j + 1]]
        rs_up_list.append(rs_up_temp)
        rs_down_list.append(rs_down_temp)
    return (rs_up_list, rs_down_list)


In [ ]:
bundle_assoc = [[0, 1], [0, 2], [0, 3], [0, 4], [1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]

def compute_charges(CohSigns):
    """Construct perturbative singlet charges from the reduced CohSigns rows."""
    charges_list = []
    for i in range(len(CohSigns)):
        charges = []
        for j in range(10):
            if CohSigns[i][j][1] != 0:
                charge_temp = [0, 0, 0, 0, 0]
                charge_temp[bundle_assoc[j][0]] = 1
                charge_temp[bundle_assoc[j][1]] = -1
                charges.append(charge_temp)
            if CohSigns[i][j][2] != 0:
                charge_temp = [0, 0, 0, 0, 0]
                charge_temp[bundle_assoc[j][0]] = -1
                charge_temp[bundle_assoc[j][1]] = 1
                charges.append(charge_temp)
        charges_list.append(charges)
    return charges_list


## Load the catalogue inputs

These files must describe the same models in the same order: `ks.json`, `CohSigns.json`, `fieldcharges.json`, and `labels.json`. The `CohSigns` preprocessing removes a fixed set of rows from each record.

`higgsnum.json` supplies the single-Higgs filter. All model arrays are filtered together.

In [ ]:
if not USE_TEST_MODEL:
    with open('ks.json', 'r') as file:
        Phi_charges = json.load(file)
    with open('CohSigns.json', 'r') as file:
        CohSigns = json.load(file)
    for i in range(len(CohSigns)):
        CohSigns[i] = np.delete(CohSigns[i], [4, 8, 9, 12, 13, 14, 16, 17, 18, 19], 0)
    phi_charges = compute_charges(CohSigns)
    with open('fieldcharges.json', 'r') as file:
        fields_charge = json.load(file)
    with open('labels.json', 'r') as file:
        labels = json.load(file)
    with open('higgsnum.json') as file:
        higgs_num = json.load(file)
    selected = [index for index, count in enumerate(higgs_num) if count == 1]
    Phi_charges = [Phi_charges[index] for index in selected]
    phi_charges = [phi_charges[index] for index in selected]
    fields_charge = [fields_charge[index] for index in selected]
    labels = [labels[index] for index in selected]
    higgs_num = [1] * len(selected)


## Optional source example

This explicit model is used only when `USE_TEST_MODEL = True`. It demonstrates charge enumeration; it is not the recovered catalogue or a validated phenomenological model.

In [ ]:
if USE_TEST_MODEL:
    Phi_charges = [[[-2, 1, 1, 0, 0], [1, -2, 0, 1, 0], [0, 1, -2, 0, 1], [1, 0, 0, -1, 0], [0, 0, 1, 0, -1]]]
    fields_charge = [[[[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0]], [[1, 0, 1, 0, 0], [1, 0, 0, 0, 1], [0, 0, 1, 1, 0]], [[0, 0, 0, 1, 1], [0, 0, 0, -1, -1]]]]
    phi_charges = [[[1, 0, -1, 0, 0], [1, 0, 0, -1, 0], [-1, 0, 0, 1, 0], [1, 0, 0, 0, -1], [0, 1, 0, -1, 0], [0, 1, 0, 0, -1], [0, -1, 1, 0, 0], [0, 0, 1, -1, 0], [0, 0, 1, 0, -1]]]
    higgs_num = [1]
    labels = ['test']


## Project charges and report search size

The shared cutoff `P` applies to all operator families. Enumeration uses only the required singlet dimensions and streams candidates. The count is $\binom{n+P}{P}$ per charge: streaming limits memory use but the search can still be expensive.

In [ ]:
n_of_models = len(Phi_charges)
QT_hlist = []
f = 5
for i in range(n_of_models):
    Q = np.concatenate((Phi_charges[i], phi_charges[i]), axis=0)
    Q_hat = linear_map(Q, 'Q')
    QT_hlist.append(np.array(Q_hat).transpose())


In [ ]:
for n in sorted({matrix.shape[1] for matrix in QT_hlist}):
    print(f'{n} singlets, P={P}: {math.comb(n + P, P):,} candidates per charge')


## Construct Yukawa insertions

Insertions are grouped by the family-index pairs they support. The strings `up yes` and `down yes` indicate coverage through the finite search cutoff; they do not establish a successful phenomenological fit.

In [ ]:
rs = generate_Yukawa_charges(fields_charge)


In [ ]:
library = {}
for i in range(n_of_models):
    library[str(labels[i])] = [Phi_charges[i], phi_charges[i], fields_charge[i]]
start = time.time()
for i in range(n_of_models):
    if i % 10 == 0:
        print('Progress:', i, ' / ', n_of_models, '\n\n')
    all_comb = len(rs[0][i])
    flag = [False] * all_comb
    c = 0
    for j in rs[0][i]:
        for p in generate_vectors(QT_hlist[i].shape[1], P):
            rhs = [-x for x in string_to_list(j)]
            if list(QT_hlist[i] @ p) == list(rhs):
                library[str(labels[i])].append(['up', p, rs[0][i][j]])
                flag[c] = True
        c += 1
    if flag == [True] * all_comb:
        library[str(labels[i])].append('up yes')
    all_comb = len(rs[1][i])
    flag = [False] * all_comb
    c = 0
    for j in rs[1][i]:
        for p in generate_vectors(QT_hlist[i].shape[1], P):
            rhs = [-x for x in string_to_list(j)]
            if list(QT_hlist[i] @ p) == list(rhs):
                library[str(labels[i])].append(['down', p, rs[1][i][j]])
                flag[c] = True
        c += 1
    if flag == [True] * all_comb:
        library[str(labels[i])].append('down yes')
end = time.time()
check_time = end - start
print('Total time:', check_time)


## Attach the Higgs-pair count

The catalogue has already been restricted to one Higgs pair per model.

In [ ]:
j = 0
for i in library:
    library[i].append(higgs_num[j])
    j += 1


In [ ]:
deleting_keys = []
for i in library:
    if [] in library[i][2]:
        deleting_keys.append(i)
for i in deleting_keys:
    del library[i]


## Save the combined library

The same in-memory library is used in the subsequent stages. Catalogue and example outputs have separate destinations.

In [ ]:
output_dir = Path('results/single_higgs_example') if USE_TEST_MODEL else Path('.')
output_dir.mkdir(parents=True, exist_ok=True)
with (output_dir / 'library_ALL_v4.json').open('w') as file:
    json.dump(library, file, indent=2)


## Construct charges for the additional operators

`HL`, `FFT`, and `FTTT` denote the field combinations used by this code. The tensor helpers convert one-based family labels in the records to zero-based Python indices. The `lambda` names are legacy code labels; contraction conventions still need project documentation.

In [ ]:
def generate_R_parity_violating_charges(charges):
    """Group HL, FFT, and FTTT index combinations by projected charge."""
    FTTT_charge = []
    FFT_charge = []
    HL_charge = []
    for i in range(len(charges)):
        HL_charge_temp = {}
        FFT_charge_temp = {}
        FTTT_charge_temp = {}
        fives = charges[i][1]
        tens = charges[i][0]
        higgs_up = charges[i][2][1]
        for j in range(len(fives)):
            charge = linear_map(list(np.array(fives[j]) + np.array(higgs_up)), 'r').tolist()
            if str(charge) in HL_charge_temp:
                HL_charge_temp[str(charge)].append([j + 1])
            else:
                HL_charge_temp[str(charge)] = [[j + 1]]
        HL_charge.append(HL_charge_temp)
        for j in range(len(fives)):
            for k in range(j, len(fives)):
                for l in range(len(tens)):
                    charge = linear_map(list(np.array(fives[j]) + np.array(fives[k]) + np.array(tens[l])), 'r').tolist()
                    if str(charge) in FFT_charge_temp:
                        FFT_charge_temp[str(charge)].append([j + 1, k + 1, l + 1])
                    else:
                        FFT_charge_temp[str(charge)] = [[j + 1, k + 1, l + 1]]
        FFT_charge.append(FFT_charge_temp)
        for j in range(len(fives)):
            for k in range(len(tens)):
                for l in range(k, len(tens)):
                    for m in range(l, len(tens)):
                        charge = linear_map(list(np.array(fives[j]) + np.array(tens[k]) + np.array(tens[l]) + np.array(tens[m])), 'r').tolist()
                        if str(charge) in FTTT_charge_temp:
                            FTTT_charge_temp[str(charge)].append([j + 1, k + 1, l + 1, m + 1])
                        else:
                            FTTT_charge_temp[str(charge)] = [[j + 1, k + 1, l + 1, m + 1]]
        FTTT_charge.append(FTTT_charge_temp)
    return (HL_charge, FFT_charge, FTTT_charge)


In [ ]:
def generate_mu_term(charges):
    """Construct projected charges of the physical Higgs bilinear."""
    mu_charges = []
    for model in charges:
        higgs_down, higgs_up = model[2][:2]
        charge = linear_map(list(np.array(higgs_down) + np.array(higgs_up)), 'r').tolist()
        mu_charges.append({str(charge): [[0, 0]]})
    return mu_charges


In [ ]:
def sort_by_P(lst, n_of_Phi):
    """Order insertions by weighted degree, using the configured VEV ranges."""
    non_perturbative_weight = math.ceil(np.emath.logn((pert_range[0] + pert_range[1]) * 0.5, (non_pert_range[0] + non_pert_range[1]) * 0.5))
    orders = []
    for i in range(len(lst)):
        tot_order = 0
        for j in range(len(lst[i][0])):
            if j < n_of_Phi:
                tot_order += non_perturbative_weight * lst[i][0][j]
            else:
                tot_order += lst[i][0][j]
        orders.append(tot_order)
    sorted_elements = [x for _, x in sorted(zip(orders, lst), key=lambda pair: pair[0], reverse=False)]
    return sorted_elements


In [ ]:
def lambda_prime(data):
    """Place FTTT exponent vectors in a tensor with four family indices."""
    max_index = 3
    lambda_list = [[[[[] for _ in range(max_index)] for _ in range(max_index)] for _ in range(max_index)] for _ in range(max_index)]
    for entry in data:
        values, indices_list = entry
        for indices in indices_list:
            p, q, r, s = [i - 1 for i in indices]
            lambda_list[p][q][r][s].append(values)
    return lambda_list

def lambda_unprime(data):
    """Place FFT exponent vectors in a tensor with three family indices."""
    max_index = 3
    lambda_list = [[[[] for _ in range(max_index)] for _ in range(max_index)] for _ in range(max_index)]
    for entry in data:
        values, indices_list = entry
        for indices in indices_list:
            p, q, r = [i - 1 for i in indices]
            lambda_list[p][q][r].append(values)
    return lambda_list

def rho(data):
    """Place HL exponent vectors in lists indexed by lepton family."""
    max_index = 3
    rho_list = [[] for _ in range(max_index)]
    for entry in data:
        values, indices_list = entry
        for indices in indices_list:
            p = indices[0] - 1
            rho_list[p].append(values)
    return rho_list

def mu(data):
    """Extract exponent vectors from single-Higgs bilinear insertion records."""
    mu_list = []
    for i in range(len(data)):
        mu_list.append(data[i][0])
    return mu_list


## Enumerate all three operator families

Each model gets `rho`, `lambda`, and `lambda_prime` tensors, including correctly shaped empty entries when no insertion is found. All searches use the same explicit cutoff `P`.

In [ ]:
operator_charges = generate_R_parity_violating_charges(fields_charge)
converters = (rho, lambda_unprime, lambda_prime)
R_parity_library = {}
for index in range(n_of_models):
    model_id = str(labels[index])
    if model_id not in library:
        continue
    families = []
    for charges, convert in zip(operator_charges, converters):
        records = []
        for charge, family_indices in charges[index].items():
            rhs = -np.array(string_to_list(charge))
            for exponent in generate_vectors(QT_hlist[index].shape[1], P):
                if np.array_equal(QT_hlist[index] @ exponent, rhs):
                    records.append([exponent, family_indices])
        families.append(convert(sort_by_P(records, len(Phi_charges[index]))))
    R_parity_library[model_id] = families


In [ ]:
with (output_dir / 'R_parity_library2.json').open('w') as file:
    json.dump(R_parity_library, file, indent=2)


## Generate single-Higgs mu insertions

`EXCLUDE_CONSTANT_MU = True` preserves the historical removal of the all-zero exponent vector. That vector represents 1: its exclusion is a physical assumption about the constant coefficient, not a consequence of zeroing singlet VEVs. Set the option to `False` to retain it.

In [ ]:
mucharges = generate_mu_term(fields_charge)
start = time.time()
mu_term_library = {}
for i in range(n_of_models):
    if str(labels[i]) in library:
        print('Progress:', i, ' / ', n_of_models, '\n\n')
        mu_term_library[str(labels[i])] = []
        for j in mucharges[i]:
            for p in generate_vectors(QT_hlist[i].shape[1], P):
                rhs = [-x for x in string_to_list(j)]
                if list(QT_hlist[i] @ p) == list(rhs):
                    if str(labels[i]) in mu_term_library:
                        mu_term_library[str(labels[i])].append([p, mucharges[i][j]])
                    else:
                        mu_term_library[str(labels[i])] = [[p, mucharges[i][j]]]
end = time.time()
check_time = end - start
print('Total time:', check_time)


In [ ]:
for i in mu_term_library:
    n_of_Phi = len(library[i][0])
    n_of_phi = len(library[i][1])
    mu_term_library[i] = sort_by_P(mu_term_library[i], n_of_Phi)
    mu_term_library[i] = mu(mu_term_library[i])
    if EXCLUDE_CONSTANT_MU:
        mu_term_library[i] = [v for v in mu_term_library[i] if any(v)]


In [ ]:
with (output_dir / 'mu_term_library2.json').open('w') as file:
    json.dump(mu_term_library, file, indent=2)


## Prepare single-Higgs batch jobs

Partition every model across `N_BATCHES` JSON files in `mu_terms_sublists/`, replacing previous batch files on reruns. Run `compute_killed_vevs0.py` for each job before opening the merge notebook.

In [ ]:
batch_dir = output_dir / 'mu_terms_sublists'
batch_dir.mkdir(parents=True, exist_ok=True)
items = list(mu_term_library.items())
for job in range(N_BATCHES):
    batch = dict(items[job::N_BATCHES])
    with (batch_dir / f'list{job}.json').open('w') as file:
        json.dump(batch, file, indent=2)
